In [0]:
# ============================================================
# Notebook: 04_accounts_cdc
# Purpose : Simulate CDC changes for account dimension and
#           apply Insert / Update / Delete using MERGE.
# ============================================================

from pyspark.sql.functions import col, lit, rand, when, current_timestamp
from pyspark.sql.types import IntegerType

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DIM_ACCOUNTS_TABLE = "data_engineering.silver_layer.dim_accounts"

# ------------------------------------------------------------
# 2. Read existing account dimension
# ------------------------------------------------------------

accounts_df = spark.table(DIM_ACCOUNTS_TABLE)

print("Current account dimension count:")
print(accounts_df.count())

# ------------------------------------------------------------
# 3. Pick sample accounts for CDC simulation
#    We are selecting a small percentage of accounts to simulate
#    changes coming from source system.
# ------------------------------------------------------------

sample_accounts_df = accounts_df.sample(fraction=0.01, seed=42)

print("CDC sample count:")
print(sample_accounts_df.count())

# ------------------------------------------------------------
# 4. Create CDC events
#    U = update existing account
#    D = logically delete / close account
# ------------------------------------------------------------

cdc_events_df = sample_accounts_df.withColumn(
    "operation",
    when(rand(seed=10) < 0.80, lit("U")).otherwise(lit("D"))
).withColumn(
    "status",
    when(col("operation") == "D", lit("CLOSED"))
    .otherwise(
        when(rand(seed=20) < 0.70, lit("ACTIVE"))
        .when(rand(seed=30) < 0.90, lit("SUSPENDED"))
        .otherwise(lit("CLOSED"))
    )
).withColumn(
    "credit_score",
    when(
        col("operation") == "U",
        (rand(seed=40) * 550 + 300).cast(IntegerType())
    ).otherwise(col("credit_score"))
).withColumn(
    "kyc_verified",
    when(
        col("operation") == "U",
        when(rand(seed=50) < 0.85, lit(True)).otherwise(lit(False))
    ).otherwise(col("kyc_verified"))
).withColumn(
    "is_deleted",
    when(col("operation") == "D", lit(True)).otherwise(lit(False))
).withColumn(
    "updated_at",
    current_timestamp()
)

# ------------------------------------------------------------
# 5. Create temp view for SQL MERGE
# ------------------------------------------------------------

cdc_events_df.createOrReplaceTempView("account_cdc_events")

print("Sample CDC events:")
cdc_events_df.display()

# ------------------------------------------------------------
# 6. Apply CDC changes using MERGE
#    If account exists:
#       - update status, credit_score, kyc_verified, is_deleted
#    If account does not exist:
#       - insert as new account
# ------------------------------------------------------------

spark.sql(f"""
MERGE INTO data_engineering.silver_layer.dim_accounts AS target
USING account_cdc_events AS source
ON target.account_id = source.account_id

WHEN MATCHED AND source.operation = 'D' THEN UPDATE SET
    target.status = 'CLOSED',
    target.is_deleted = true,
    target.updated_at = source.updated_at

WHEN MATCHED AND source.operation = 'U' THEN UPDATE SET
    target.status = source.status,
    target.credit_score = source.credit_score,
    target.kyc_verified = source.kyc_verified,
    target.updated_at = source.updated_at

WHEN NOT MATCHED AND source.operation = 'I' THEN INSERT (
    account_id,
    account_type,
    status,
    credit_score,
    kyc_verified,
    opened_date,
    is_deleted,
    created_at,
    updated_at
)
VALUES (
    source.account_id,
    source.account_type,
    source.status,
    source.credit_score,
    source.kyc_verified,
    source.opened_date,
    false,
    source.created_at,
    source.updated_at
)
""")

print("CDC MERGE completed successfully.")

# ------------------------------------------------------------
# 7. Validation
# ------------------------------------------------------------

print("Account status summary:")
spark.sql(f"""
SELECT
    status,
    is_deleted,
    COUNT(*) AS account_count
FROM {DIM_ACCOUNTS_TABLE}
GROUP BY status, is_deleted
ORDER BY status, is_deleted
""").display()

print("Deleted / closed account count:")
spark.sql(f"""
SELECT COUNT(*) AS deleted_accounts
FROM {DIM_ACCOUNTS_TABLE}
WHERE is_deleted = true
""").display()

print("Sample updated accounts:")
spark.sql(f"""
SELECT *
FROM {DIM_ACCOUNTS_TABLE}
WHERE is_deleted = true
LIMIT 20
""").display()